# Sprint 3 — Pipeline Overhaul Validation

**Re-runs Sprint 2's exact 10 questions** to measure improvement from:

| Change | What it fixes |
|---|---|
| Enriched catalog (themes, when_to_use, use_cases) | Better template matching |
| Fixed selection prompt (removed NLP/sentiment bias) | Stops irrelevant template picks |
| Rewritten integration prompt (answer-focused) | Fixes incomplete outputs |
| Max 2-3 templates (was 4-5) | Reduces token competition |
| Template content in integrator summaries | Smarter integration |
| Complexity router (assess_complexity + smart_execute) | Routes simple questions to raw |

## Test Conditions

| Condition | Description |
|---|---|
| **A. Raw** | Same as Sprint 2 — bare question, no template |
| **B. Smart (new)** | Uses `smart_execute()` — the complexity router decides |
| **C. Integrated (v2)** | Uses the improved pipeline (enriched catalog + new prompts + max 3 templates) |

**Key hypothesis**: Smart execute should match or beat raw on simple questions AND beat raw on complex ones.

---

**Estimated cost**: ~$0.80-1.50 | **Time**: ~15-20 min

In [ ]:
import json
import os
import sys
import time
from collections import defaultdict
from datetime import datetime

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..'))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

# os.environ['OPENAI_API_KEY'] = 'sk-...'

PROVIDER = 'openai'
EVAL_MODE = 'llm'
print(f'Repo root: {REPO_ROOT}')
print(f'Eval mode: {EVAL_MODE}')

In [ ]:
from mycontext.core import Context
from mycontext.foundation import Directive
from mycontext.intelligence.output_evaluator import OutputEvaluator
from mycontext.intelligence.pattern_suggester import (
    assess_complexity,
    smart_execute,
)
from mycontext.intelligence.template_integrator_agent import TemplateIntegratorAgent

evaluator = OutputEvaluator(mode=EVAL_MODE, provider=PROVIDER)
integrator = TemplateIntegratorAgent(include_enterprise=True)

print('All components loaded (with pipeline overhaul).')

## Same 10 Questions from Sprint 2

In [ ]:
TEST_QUESTIONS = [
    "Why did customer churn spike 40% last quarter and what should we do about it?",
    "Should we migrate our monolithic architecture to microservices? What are the trade-offs?",
    "Our AI model is producing biased hiring recommendations. How do we diagnose and fix this?",
    "Design a go-to-market strategy for a B2B SaaS product entering the healthcare vertical.",
    "Our API response times tripled after the last deployment. Find the root cause and propose fixes.",
    "How should a startup allocate its $2M seed funding across engineering, marketing, and operations?",
    "Evaluate the ethical implications of using facial recognition in public schools.",
    "Our cross-functional team has persistent communication breakdowns. Diagnose and solve.",
    "Compare three database options (PostgreSQL, MongoDB, DynamoDB) for a real-time analytics platform.",
    "Develop a 12-month roadmap for transitioning from waterfall to agile across a 200-person org.",
]

# Sprint 2 scores for comparison
S2_RAW = [0.90, 0.92, 0.94, 0.94, 1.00, 0.92, 0.88, 0.98, 0.92, 0.815]
S2_INT = [0.8975, 0.88, 0.96, 0.86, 0.65, 0.94, 0.82, 0.96, 0.88, 0.86]

print(f'{len(TEST_QUESTIONS)} questions ready')

## Helper Functions

In [ ]:
def execute_context(ctx, provider=PROVIDER):
    t0 = time.time()
    result = ctx.execute(provider=provider)
    return result.response, time.time() - t0


def format_dims(score):
    short = {'instruction_following': 'IF', 'reasoning_depth': 'RD',
             'actionability': 'AC', 'structure_compliance': 'SC',
             'cognitive_scaffolding': 'CS'}
    parts = [f'{short.get(d.value, d.value)}={v:.0%}' for d, v in score.dimensions.items()]
    return f'{score.overall:.1%} [{" ".join(parts)}]'


print('Helpers ready.')

## Step 1: Complexity Assessment (Preview)

Before running the full test, let's see what the complexity router recommends for each question.

In [ ]:
assessments = []
print(f'{"#":<3} {"Complexity":>10} {"Recommendation":>16} {"Domains":>35} {"Best Template":>30}')
print('-' * 100)

for i, q in enumerate(TEST_QUESTIONS):
    a = assess_complexity(q, provider=PROVIDER)
    assessments.append(a)
    domains = ', '.join(a.domains[:3])
    print(f'{i+1:<3} {a.complexity:>10} {a.recommendation:>16} {domains:>35} {(a.best_template or "-"):>30}')

recs = [a.recommendation for a in assessments]
print(f'\nRouting: {recs.count("raw")} raw, {recs.count("single_template")} single, {recs.count("integrated")} integrated')

---

## Step 2: Full Test — 3 Conditions x 10 Questions

**Expect ~2-3 min per question. Total: ~20-30 min.**

In [ ]:
results = []

for i, question in enumerate(TEST_QUESTIONS):
    assessment = assessments[i]
    print(f'\n{"="*70}')
    print(f'[{i+1}/10] {question[:65]}...')
    print(f'Router: {assessment.recommendation} (complexity={assessment.complexity})')
    print('='*70)
    row = {'question': question, 'assessment': assessment.to_dict()}

    # --- A: Raw (same as Sprint 2) ---
    print('  [A] Raw...', end=' ', flush=True)
    try:
        ctx_a = Context(directive=Directive(content=question))
        out_a, t_a = execute_context(ctx_a)
        score_a = evaluator.evaluate(ctx_a, out_a)
        row['raw'] = {'output': out_a, 'score': score_a, 'time': t_a}
        print(format_dims(score_a))
    except Exception as e:
        row['raw'] = {'error': str(e)}
        print(f'ERROR: {e}')

    # --- B: Smart Execute (new complexity router) ---
    print(f'  [B] Smart ({assessment.recommendation})...', end=' ', flush=True)
    try:
        t0 = time.time()
        out_b, meta_b = smart_execute(question, provider=PROVIDER)
        t_b = time.time() - t0
        ctx_b_eval = Context(directive=Directive(content=question))
        score_b = evaluator.evaluate(ctx_b_eval, out_b)
        row['smart'] = {
            'output': out_b, 'score': score_b, 'time': t_b,
            'mode': meta_b.get('mode', ''),
            'templates_used': meta_b.get('templates_used', []),
        }
        tpls = meta_b.get('templates_used', [])
        print(f'{format_dims(score_b)}  mode={meta_b["mode"]} templates={tpls}')
    except Exception as e:
        row['smart'] = {'error': str(e)}
        print(f'ERROR: {e}')

    # --- C: Integrated v2 (improved pipeline, max 3 templates) ---
    print('  [C] Integrated v2...', end=' ', flush=True)
    try:
        t0 = time.time()
        integration = integrator.suggest_and_integrate(
            question=question, provider=PROVIDER, max_patterns=3,
            integration_mode='full',
        )
        ctx_c = integration.to_context()
        out_c, _ = execute_context(ctx_c)
        t_c = time.time() - t0
        score_c = evaluator.evaluate(ctx_c, out_c)
        row['integrated_v2'] = {
            'output': out_c, 'score': score_c, 'time': t_c,
            'source_templates': integration.source_templates,
        }
        print(f'{format_dims(score_c)}  templates={integration.source_templates}')
    except Exception as e:
        row['integrated_v2'] = {'error': str(e)}
        print(f'ERROR: {e}')

    results.append(row)

print(f'\n\nDone! All {len(results)} questions tested.')

## Results Summary

In [ ]:
CONDITIONS = [('raw', 'Raw'), ('smart', 'Smart'), ('integrated_v2', 'IntV2')]

print(f'{"#":<3} {"Question":<40} {"Raw":>6} {"Smart":>7} {"IntV2":>7} {"S2Raw":>7} {"S2Int":>7} {"Winner":>8} {"Router":>12}')
print('-' * 105)

wins = {label: 0 for _, label in CONDITIONS}

for i, row in enumerate(results):
    q = row['question'][:38]
    scores = {}
    for cond, label in CONDITIONS:
        s = row.get(cond, {}).get('score')
        scores[label] = s.overall if s else None

    strs = {label: (f'{v:.0%}' if v is not None else 'ERR') for label, v in scores.items()}
    valid = {k: v for k, v in scores.items() if v is not None}
    if valid:
        winner = max(valid, key=valid.get)
        wins[winner] += 1
    else:
        winner = '?'

    router = row.get('assessment', {}).get('recommendation', '?')
    s2r = f'{S2_RAW[i]:.0%}'
    s2i = f'{S2_INT[i]:.0%}'

    print(f'{i+1:<3} {q:<40} {strs["Raw"]:>6} {strs["Smart"]:>7} {strs["IntV2"]:>7} {s2r:>7} {s2i:>7} {winner:>8} {router:>12}')

print('-' * 105)
total = sum(wins.values())
for _, label in CONDITIONS:
    print(f'{label}: {wins[label]}/{total} wins ({wins[label]/max(total,1):.0%})', end='  |  ')
print()

## Per-Dimension Averages

In [ ]:
dim_data = defaultdict(lambda: {c: [] for c, _ in CONDITIONS})
overall_data = {c: [] for c, _ in CONDITIONS}

for row in results:
    for cond, _ in CONDITIONS:
        s = row.get(cond, {}).get('score')
        if s:
            overall_data[cond].append(s.overall)
            for dim, val in s.dimensions.items():
                dim_data[dim.value][cond].append(val)

def avg(lst):
    return sum(lst) / len(lst) if lst else 0

print(f'{"Dimension":<25} {"Raw":>7} {"Smart":>8} {"IntV2":>8}')
print('-' * 55)

for dim_name in ['instruction_following', 'reasoning_depth', 'actionability',
                 'structure_compliance', 'cognitive_scaffolding']:
    r = avg(dim_data[dim_name]['raw'])
    sm = avg(dim_data[dim_name]['smart'])
    iv = avg(dim_data[dim_name]['integrated_v2'])
    label = dim_name.replace('_', ' ').title()
    print(f'{label:<25} {r:>6.1%} {sm:>7.1%} {iv:>7.1%}')

print('-' * 55)
r_o = avg(overall_data['raw'])
sm_o = avg(overall_data['smart'])
iv_o = avg(overall_data['integrated_v2'])
print(f'{"OVERALL":<25} {r_o:>6.1%} {sm_o:>7.1%} {iv_o:>7.1%}')

print(f'\nSprint 2 averages: Raw={sum(S2_RAW)/len(S2_RAW):.1%}, Integrated={sum(S2_INT)/len(S2_INT):.1%}')
print(f'Sprint 3 averages: Raw={r_o:.1%}, Smart={sm_o:.1%}, IntV2={iv_o:.1%}')

## Sprint 2 vs Sprint 3 — Side-by-Side

In [ ]:
print(f'{"#":<3} {"S2 Raw":>8} {"S3 Raw":>8} {"S2 Int":>8} {"S3 IntV2":>9} {"S3 Smart":>9} {"Int Δ":>7} {"Smart vs Raw":>13}')
print('-' * 75)

s2_int_total = 0
s3_int_total = 0

for i in range(len(results)):
    s3_raw = results[i].get('raw', {}).get('score')
    s3_smart = results[i].get('smart', {}).get('score')
    s3_int = results[i].get('integrated_v2', {}).get('score')

    s3r = s3_raw.overall if s3_raw else 0
    s3s = s3_smart.overall if s3_smart else 0
    s3i = s3_int.overall if s3_int else 0

    int_delta = s3i - S2_INT[i]
    smart_vs_raw = s3s - s3r

    s2_int_total += S2_INT[i]
    s3_int_total += s3i

    print(
        f'Q{i+1:<2} '
        f'{S2_RAW[i]:>7.1%} {s3r:>7.1%} '
        f'{S2_INT[i]:>7.1%} {s3i:>8.1%} {s3s:>8.1%} '
        f'{int_delta:>+6.1%} {smart_vs_raw:>+12.1%}'
    )

print('-' * 75)
print(f'\nIntegration improvement (S2 → S3): {(s3_int_total/10 - s2_int_total/10):+.1%} avg')
print('Positive Int Δ = pipeline overhaul improved integration quality.')
print('Positive Smart vs Raw = smart_execute outperformed raw prompt.')

## Router Decision Analysis

Did the complexity router make correct decisions?

In [ ]:
print(f'{"#":<3} {"Router":>16} {"Smart Score":>12} {"Raw Score":>10} {"Smart Won?":>11} {"Correct Route?":>15}')
print('-' * 75)

correct = 0
for i, row in enumerate(results):
    rec = row.get('assessment', {}).get('recommendation', '?')
    s_smart = row.get('smart', {}).get('score')
    s_raw = row.get('raw', {}).get('score')
    s_int = row.get('integrated_v2', {}).get('score')

    smart_val = s_smart.overall if s_smart else 0
    raw_val = s_raw.overall if s_raw else 0
    int_val = s_int.overall if s_int else 0

    smart_won = smart_val >= raw_val
    best_option = max([('raw', raw_val), ('smart', smart_val), ('int', int_val)], key=lambda x: x[1])

    # Correct if: raw recommended and raw was best, OR template recommended and template beat raw
    if rec == 'raw' and raw_val >= max(smart_val, int_val):
        correct_route = 'YES'
        correct += 1
    elif rec in ('single_template', 'integrated') and smart_val >= raw_val:
        correct_route = 'YES'
        correct += 1
    else:
        correct_route = 'NO'

    print(f'Q{i+1:<2} {rec:>16} {smart_val:>11.1%} {raw_val:>9.1%} {"YES" if smart_won else "NO":>11} {correct_route:>15}')

print(f'\nRouter accuracy: {correct}/{len(results)} ({correct/max(len(results),1):.0%})')

## Deep Dive — Inspect Any Question

In [ ]:
IDX = 0  # Change to 0-9

row = results[IDX]
print(f'Question: {row["question"]}\n')
print(f'Router: {row["assessment"]}\n')

for cond, label in [('raw', 'A. RAW'), ('smart', 'B. SMART'), ('integrated_v2', 'C. INTEGRATED V2')]:
    data = row.get(cond, {})
    if 'error' in data:
        print(f'\n--- {label} --- ERROR: {data["error"]}\n')
        continue
    score = data.get('score')
    print(f'\n{"="*70}')
    print(f'{label} — {format_dims(score)}')
    if cond == 'smart':
        print(f'Mode: {data.get("mode", "?")} | Templates: {data.get("templates_used", [])}')
    if cond == 'integrated_v2' and 'source_templates' in data:
        print(f'Templates: {" → ".join(data["source_templates"])}')
    if score:
        print(f'Strengths: {", ".join(score.strengths[:3]) if score.strengths else "N/A"}')
        print(f'Weaknesses: {", ".join(score.weaknesses[:3]) if score.weaknesses else "N/A"}')
    print('='*70)
    output = data.get('output', '')
    print(output[:2000])
    if len(output) > 2000:
        print(f'\n... ({len(output)} total chars)')

## Save Results

In [ ]:
def serialize(results):
    out = []
    for row in results:
        r = {'question': row['question'], 'assessment': row.get('assessment', {})}
        for cond in ['raw', 'smart', 'integrated_v2']:
            data = row.get(cond, {})
            if 'error' in data:
                r[cond] = {'error': data['error']}
            elif 'score' in data:
                s = data['score']
                r[cond] = {
                    'overall': round(s.overall, 4),
                    'dimensions': {d.value: round(v, 4) for d, v in s.dimensions.items()},
                    'time_seconds': round(data.get('time', 0), 2),
                    'output_length': len(data.get('output', '')),
                }
                if 'source_templates' in data:
                    r[cond]['source_templates'] = data['source_templates']
                if 'templates_used' in data:
                    r[cond]['templates_used'] = data['templates_used']
                if 'mode' in data:
                    r[cond]['smart_mode'] = data['mode']
                if s.strengths:
                    r[cond]['strengths'] = s.strengths[:3]
                if s.weaknesses:
                    r[cond]['weaknesses'] = s.weaknesses[:3]
        out.append(r)
    return out

report = {
    'sprint': 'Sprint 3 — Pipeline Overhaul Validation',
    'date': datetime.now().isoformat(),
    'provider': PROVIDER,
    'eval_mode': EVAL_MODE,
    'num_questions': len(results),
    'changes': [
        'Enriched catalog (themes, when_to_use, use_cases)',
        'Fixed selection prompt (removed NLP bias, business example)',
        'Rewritten integration prompt (answer-focused, 5-7 sections max)',
        'Max 3 templates (was 5)',
        'Template content in integrator summaries',
        'Complexity router (assess_complexity + smart_execute)',
    ],
    'summary': {
        'avg_overall': {c: round(avg(overall_data[c]), 4) for c, _ in CONDITIONS},
        'wins': dict(wins),
        's2_comparison': {
            's2_raw_avg': round(sum(S2_RAW) / len(S2_RAW), 4),
            's2_int_avg': round(sum(S2_INT) / len(S2_INT), 4),
        },
    },
    'results': serialize(results),
}

outpath = 'sprint3_pipeline_results.json'
with open(outpath, 'w') as f:
    json.dump(report, f, indent=2)

print(f'Results saved to {outpath}')